# Person Detection Demo (YOLOv8) — Google Colab (GPU T4)

เป้าหมาย: สาธิตการตรวจจับ **บุคคล** (person) จากภาพ/วิดีโอ โดยใช้ YOLOv8 (Ultralytics)

## ก่อนเริ่ม
ไปที่ **Runtime → Change runtime type → GPU** (แนะนำ T4)

> COCO class id = 0 คือ `person` จึงใช้ `classes=[0]` เพื่อกรองเฉพาะบุคคล


In [ ]:
# 1) Install
!pip -q install ultralytics opencv-python
import ultralytics
ultralytics.checks()


In [ ]:
# 2) Verify GPU (ต้องเห็น CUDA=True)
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# 3) Load model (เลือก n หรือ s)
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
model


## 4) Inference on an image
อัปโหลดภาพของคุณเอง (แนะนำ: เบลอหน้า/ไม่ระบุตัวตน)


In [ ]:
from google.colab import files
uploaded = files.upload()
img_path = next(iter(uploaded.keys()))
img_path


In [ ]:
# Run prediction (person only) on GPU (device=0)
results = model.predict(source=img_path, classes=[0], conf=0.25, device=0, save=True)
import matplotlib.pyplot as plt
img = results[0].plot()  # numpy array with boxes
plt.figure(figsize=(10,6))
plt.imshow(img)
plt.axis('off')
plt.show()


## 5) Inference on a video
อัปโหลดวิดีโอสั้น 10–30 วินาที เพื่อให้รันเร็วและโชว์ผลทัน


In [ ]:
uploaded = files.upload()
vid_path = next(iter(uploaded.keys()))
vid_path


In [ ]:
# Predict video and display output
results = model.predict(source=vid_path, classes=[0], conf=0.25, device=0, save=True)
import glob, os
from IPython.display import Video, display

# Find latest output video under runs/detect/
cands = sorted(glob.glob('runs/detect/predict*/**/*.*', recursive=True))
video_out = None
for p in reversed(cands):
    if os.path.splitext(p)[1].lower() in ['.mp4', '.avi', '.mov', '.mkv']:
        video_out = p
        break
print('Output:', video_out)
if video_out:
    display(Video(video_out, embed=True))


## 6) (Optional) Fine-tune on a custom dataset (YOLO format)

### โครงสร้างที่คาดหวัง
```
datasets/mydata/
  images/train, images/val
  labels/train, labels/val
  data.yaml
```

แนะนำ: ทำเป็น `dataset.zip` แล้วอัปโหลด


In [ ]:
# Upload dataset.zip (optional)
# from google.colab import files
# uploaded = files.upload()
# !unzip -q dataset.zip -d datasets
# !find datasets -maxdepth 3 -type f | head
print('Skip if you do not have dataset.zip')


In [ ]:
# Fine-tune (optional) — แนะนำ epochs=1–3 สำหรับเดโม
# data_yaml = 'datasets/mydata/data.yaml'
# ft = YOLO('yolov8n.pt')
# ft.train(data=data_yaml, epochs=3, imgsz=640, batch=16, device=0)
print('Uncomment to train')


## 7) (Optional) Compare before vs after
- ก่อนสอน: ใช้ `yolov8n.pt`
- หลังสอน: ใช้ `runs/train/exp/weights/best.pt` (หรือ path ที่ได้)
แล้ว predict ภาพเดิมเพื่อให้เห็นความต่าง
